### Problem1
Ref: 

https://doi.org/10.1080/14686996.2016.1165584

https://doi.org/10.1002/advs.202101099

https://velog.io/@danielseo/Computer-Vision-DnCNN

In [ ]:
import numpy as np
import numpy.random as nr
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device: {device}")

device: cpu


In [ ]:
class imageGenerator:
    def __init__(self, N):
        self.d = 3.16 # Mo-Mo, S-S distance in 2d layer
        a = self.d/np.sqrt(3) # lattice constant of hexagonal lattice

        self.h = 0.15 # pixel size
        self.imgsize = 64 # image size

        a1 = np.array([1, 0])
        a2 = np.array([-1/2, np.sqrt(3)/2])

        self.N = N
        self.N2 = self.N*self.N
        self.atomPos = np.zeros((3, self.N2, 2), dtype=float)
        self.atomType= np.zeros((3, self.N2), dtype=int)
        # atomType == 0 Vacancy
        #          == 1 S
        #          == 2 Mo

        for i in range(self.N):
            for j in range(self.N):
                # location of Mo in the second layer
                idx = i*self.N + j
                R = i*self.d*a1 + j*self.d*a2 
                self.atomPos[1, idx, 0] = R[0]
                self.atomPos[1, idx, 1] = R[1]
                self.atomType[1, idx] = 2

                # location of S in the first layer
                R += a*np.array([0, 1]) 
                self.atomPos[0, idx, 0] = R[0]
                self.atomPos[0, idx, 1] = R[1]
                self.atomType[0, idx] = 1

                # location of S in the third layer
                self.atomPos[2, idx, 0] = R[0]
                self.atomPos[2, idx, 1] = R[1]
                self.atomType[2, idx] = 1

        cx = np.mean(self.atomPos[:, :, 0]) # center x
        cy = np.mean(self.atomPos[:, :, 1]) # center y
        self.atomPos[:, :, 0] -= cx
        self.atomPos[:, :, 1] -= cy

    
    def latticeDistortion(self, MoRate=0.05, SRate=0.4):
        # Displacement, Vacancy
        # MoRate, SRate: vacancy rate of Mo and S

        # Displacement
        displacement = nr.normal(loc=0, scale=0.02, size=(3, self.N2, 2))
        self.atomPos += displacement

        # Vacancy
        for i in range(self.N2):
            if nr.rand() < MoRate:
                # Mo vacancy
                self.atomType[1, i] = 0

            if nr.rand() < SRate:
                # S1 vacancy
                self.atomType[0, i] = 0

            if nr.rand() < SRate:
                # S2 vacancy
                self.atomType[2, i] = 0


    def makeImage(self, s_Mo=0.35, s_S=0.5, returnCrop=True):
        # Rotation
        angle = nr.rand()*2*np.pi
        pos = np.zeros_like(self.atomPos)
        pos[:, :, 0] = np.cos(angle)*self.atomPos[:, :, 0] - np.sin(angle)*self.atomPos[:, :, 1]
        pos[:, :, 1] = np.sin(angle)*self.atomPos[:, :, 0] + np.cos(angle)*self.atomPos[:, :, 1]

        self.atomPos = np.copy(pos)

        start_x = np.min(self.atomPos[:, :, 0])
        end_x = np.max(self.atomPos[:, :, 0])+self.h
        start_y = np.min(self.atomPos[:, :, 1])
        end_y = np.max(self.atomPos[:, :, 1])+self.h

        x = np.arange(start_x, end_x, self.h)
        y = np.arange(start_y, end_y, self.h)

        X, Y = np.meshgrid(x, y)

        l = self.imgsize*self.h
        imgidx_x = int((-l/2 - start_x)/self.h)
        imgidx_y = int((-l/2 - start_y)/self.h)

        occupancy = np.sign(self.atomType)
        
        # Mo
        field = occupancy[1, :][:, np.newaxis, np.newaxis]*42/(2*np.pi*s_Mo**2)*\
                np.exp(-(X - self.atomPos[1, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_Mo**2))*\
                np.exp(-(Y - self.atomPos[1, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_Mo**2))
        
        # S in the first layer
        field += occupancy[0, :][:, np.newaxis, np.newaxis]*16/(2*np.pi*s_S**2)*\
                np.exp(-(X - self.atomPos[0, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_S**2))*\
                np.exp(-(Y - self.atomPos[0, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_S**2))
        
        # S in the last layer
        field += occupancy[2, :][:, np.newaxis, np.newaxis]*16/(2*np.pi*s_S**2)*\
                np.exp(-(X - self.atomPos[2, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_S**2))*\
                np.exp(-(Y - self.atomPos[2, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_S**2))
        
        # Final field
        field = field.sum(axis=0)
        
        if returnCrop:
            return field[imgidx_y:imgidx_y+64, imgidx_x:imgidx_x+64]
        else:
            return field, imgidx_x, imgidx_y
    

def imageNoise(img):
    # Gaussian noise (thermal noise) + Poisson noise (photon noise)
    rng = nr.default_rng() # Random number generator
    size = np.shape(img)

    # Poisson noise
    img_max = np.max(img)
    scaled_img = img/img_max*8 # 8 electrons
    img_noise = rng.poisson(scaled_img)*img_max/8 

    # Gaussian noise
    img_noise += rng.normal(loc=7, scale=3, size=size) 

    return img_noise

In [ ]:
class DnCNN(nn.Module):
    def __init__(self, channels, depth):
        super().__init__()
        layers = []

        # First layer
        layers.append(nn.Conv2d(1, channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        
        # Middle layers
        for _ in range(depth-2):
            layers.append(nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(channels))
            layers.append(nn.ReLU())

        # Last layer
        layers.append(nn.Conv2d(in_channels=channels, out_channels=1, kernel_size=3, padding=1))

        self.net = nn.Sequential(*layers) # Neural network


    def forward(self, x):
        noise = self.net(x)
        return noise


class DenoiseDataset(Dataset):
    def __init__(self, clean_image, noisy_image, noise):
        self.clean_image = clean_image
        self.noisy_image = noisy_image
        self.noise = noise

    
    def __len__(self):
        return len(self.clean_image)
    

    def __getitem__(self, idx):
        return self.noisy_image[idx], self.noise[idx]

In [ ]:
N_training = 1000 # number of training images

imgs_clean = np.zeros((N_training, 1, 64, 64), dtype=np.float32)
imgs_noise = np.zeros((N_training, 1, 64, 64), dtype=np.float32) # input

print("Generating a training set...", flush=True)

for i in range(N_training):
    img_g = imageGenerator(6)
    img_g.latticeDistortion()
    imgs_clean[i, 0, :, :] = img_g.makeImage()
    imgs_noise[i, 0, :, :] = imageNoise(imgs_clean[i, 0, :, :])

print("Done", flush=True)

imgs_clean = torch.from_numpy(imgs_clean).float()
imgs_noise = torch.from_numpy(imgs_noise).float()
noise = imgs_noise - imgs_clean

dataset = DenoiseDataset(imgs_clean, imgs_noise, noise)
loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [ ]:
model = DnCNN(64, 8).to(device)

epochs = 2000
count = epochs/10

#Loss = nn.L1Loss()
Loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_list = np.zeros(epochs, dtype=float)

print("Training start", flush=True)

model.train()

for epoch in range(epochs):
    loss_per_running = 0

    for noisy_batch, noise_batch in loader:
        noisy_batch = noisy_batch.to(device)
        noise_batch = noise_batch.to(device)
        
        optimizer.zero_grad()

        pred = model(noisy_batch)
        loss = Loss(pred, noise_batch)
        loss.backward()
        loss_per_running += loss.item()
        optimizer.step()
        
    loss_list[epoch] = loss_per_running/len(loader)

    if (epoch + 1) % count == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss_list[epoch]}", flush=True)

torch.save(model.state_dict(), "DnCNN_parameters.pt")

plt.plot(loss_list)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig("loss_DnCNN.png")
plt.clf()

In [ ]:
noised_np = np.load("data/TestImages_Noised.npy")
noised_tensor = torch.from_numpy(np.transpose(noised_np, (2,0,1))[:, np.newaxis, :, :]).float().to(device) # np.array to tensor

#model = DnCNN(64, 8).to(device)
#model.load_state_dict(torch.load("DnCNN_parameters.pt", weights_only=True, map_location=torch.device('cpu')))
model.eval()
with torch.no_grad():
    denoised_tensor = noised_tensor - model(noised_tensor)

denoise_np = denoised_tensor.squeeze(1).permute(1,2,0).cpu().numpy() # tensor to np.array

np.save("DenoisedImages.npy", denoise_np)

plt.figure(figsize=(20, 6))

for i in range(10):
    noisedImage = noised_np[:, :, i]
    denoiseImage = denoise_np[:, :, i]

    plt.subplot(2, 10, 2*i + 1)
    plt.imshow(noisedImage)
    plt.axis("off")

    plt.subplot(2, 10, 2*i + 2)
    plt.imshow(denoiseImage)
    plt.axis("off")

plt.tight_layout()

plt.savefig("comparison.png", dpi=300)

### Problem 2

In [ ]:
class datasetGenerator:
    def __init__(self, N):
        self.d = 3.16 # Mo-Mo, S-S distance in 2d layer
        a = self.d/np.sqrt(3) # lattice constant of hexagonal lattice

        self.h = 0.15 # pixel size
        self.imgsize = 64 # image size

        a1 = np.array([1, 0])
        a2 = np.array([-1/2, np.sqrt(3)/2])

        self.N = N
        self.N2 = self.N*self.N
        self.atomPos = np.zeros((3, self.N2, 2), dtype=float)
        self.atomType= np.zeros((3, self.N2), dtype=int)
        # atomType == 0 Vacancy
        #          == 1 S
        #          == 2 Mo

        for i in range(self.N):
            for j in range(self.N):
                # location of Mo in the second layer
                idx = i*self.N + j
                R = i*self.d*a1 + j*self.d*a2 
                self.atomPos[1, idx, 0] = R[0]
                self.atomPos[1, idx, 1] = R[1]
                self.atomType[1, idx] = 2

                # location of S in the first layer
                R += a*np.array([0, 1]) 
                self.atomPos[0, idx, 0] = R[0]
                self.atomPos[0, idx, 1] = R[1]
                self.atomType[0, idx] = 1

                # location of S in the third layer
                self.atomPos[2, idx, 0] = R[0]
                self.atomPos[2, idx, 1] = R[1]
                self.atomType[2, idx] = 1

        cx = np.mean(self.atomPos[:, :, 0]) # center x
        cy = np.mean(self.atomPos[:, :, 1]) # center y
        self.atomPos[:, :, 0] -= cx
        self.atomPos[:, :, 1] -= cy

    
    def latticeDistortion(self, MoRate=0.05, SRate=0.4):
        # Displacement, Vacancy
        # MoRate, SRate: vacancy rate of Mo and S

        # Displacement
        displacement = nr.normal(loc=0, scale=0.02, size=(3, self.N2, 2))
        self.atomPos += displacement

        # Vacancy
        for i in range(self.N2):
            if nr.rand() < MoRate:
                # Mo vacancy
                self.atomType[1, i] = 0

            if nr.rand() < SRate:
                # S1 vacancy
                self.atomType[0, i] = 0

            if nr.rand() < SRate:
                # S2 vacancy
                self.atomType[2, i] = 0
     

    def makeImage(self, s_Mo=0.35, s_S=0.5):
        # Rotation
        angle = nr.rand()*2*np.pi
        pos = np.zeros_like(self.atomPos)
        pos[:, :, 0] = np.cos(angle)*self.atomPos[:, :, 0] - np.sin(angle)*self.atomPos[:, :, 1]
        pos[:, :, 1] = np.sin(angle)*self.atomPos[:, :, 0] + np.cos(angle)*self.atomPos[:, :, 1]

        self.atomPos = np.copy(pos)

        start_x = np.min(self.atomPos[:, :, 0])
        end_x = np.max(self.atomPos[:, :, 0])+self.h
        start_y = np.min(self.atomPos[:, :, 1])
        end_y = np.max(self.atomPos[:, :, 1])+self.h

        x = np.arange(start_x, end_x, self.h)
        y = np.arange(start_y, end_y, self.h)

        self.X, self.Y = np.meshgrid(x, y)

        l = self.imgsize*self.h
        self.imgidx_x = int((-l/2 - start_x)/self.h)
        self.imgidx_y = int((-l/2 - start_y)/self.h)

        occupancy = np.sign(self.atomType)
        
        # Mo
        field = occupancy[1, :][:, np.newaxis, np.newaxis]*42/(2*np.pi*s_Mo**2)*\
                np.exp(-(self.X - self.atomPos[1, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_Mo**2))*\
                np.exp(-(self.Y - self.atomPos[1, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_Mo**2))
        
        # S in the first layer
        field += occupancy[0, :][:, np.newaxis, np.newaxis]*16/(2*np.pi*s_S**2)*\
                 np.exp(-(self.X - self.atomPos[0, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_S**2))*\
                 np.exp(-(self.Y - self.atomPos[0, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_S**2))
        
        # S in the last layer
        field += occupancy[2, :][:, np.newaxis, np.newaxis]*16/(2*np.pi*s_S**2)*\
                 np.exp(-(self.X - self.atomPos[2, :, 0][:, np.newaxis, np.newaxis])**2/(2*s_S**2))*\
                 np.exp(-(self.Y - self.atomPos[2, :, 1][:, np.newaxis, np.newaxis])**2/(2*s_S**2))
        
        # Final cropped image
        return field.sum(axis=0)[self.imgidx_y:self.imgidx_y+64, self.imgidx_x:self.imgidx_x+64]
    

    def makeMask(self, s_Mo=0.35, s_S=0.5):
        mask = np.zeros((5, *np.shape(self.X)), dtype=float)
        '''
        mask[0]: Mo
        mask[1]: Double S
        mask[2]: Double S vacancy
        mask[3]: Single S vacancy
        mask[4]: Mo vacancy
        '''

        for i in range(self.N2):
            # Mo
            if self.atomType[1, i] == 2:
                mask[0] += np.exp(-(self.X - self.atomPos[1, i, 0])**2/(2*s_Mo**2))*\
                           np.exp(-(self.Y - self.atomPos[1, i, 1])**2/(2*s_Mo**2))
                
            # Mo vacancy
            else: 
                mask[4] += np.exp(-(self.X - self.atomPos[1, i, 0])**2/(2*s_Mo**2))*\
                           np.exp(-(self.Y - self.atomPos[1, i, 1])**2/(2*s_Mo**2))
                
            # Double S
            if (self.atomType[0, i] == 1) and (self.atomType[2, i] == 1):
                mask[1] += np.exp(-(self.X - self.atomPos[0, i, 0])**2/(2*s_S**2))*\
                           np.exp(-(self.Y - self.atomPos[0, i, 1])**2/(2*s_S**2))
                
            # Double S vacancy
            if (self.atomType[0, i] == 0) and (self.atomType[2, i] == 0):
                mask[2] += np.exp(-(self.X - self.atomPos[0, i, 0])**2/(2*s_S**2))*\
                           np.exp(-(self.Y - self.atomPos[0, i, 1])**2/(2*s_S**2))
                
            # Singe S vacancy
            if (self.atomType[0, i] == 0) ^ (self.atomType[2, i] == 0):
                mask[3] += np.exp(-(self.X - self.atomPos[0, i, 0])**2/(2*s_S**2))*\
                           np.exp(-(self.Y - self.atomPos[0, i, 1])**2/(2*s_S**2))
                

                
                
        mask = mask[:, self.imgidx_y:self.imgidx_y+64, self.imgidx_x:self.imgidx_x+64]
        mask = np.clip(mask, 0, 1)

        return mask
    

class ClassificationDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    
    def __len__(self):
        return len(self.images)


    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]

In [ ]:
class DoubleConv2d(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(output_channels, output_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)
    

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=5):
        # in_channels=1: Gray scale
        # out_channels=5: Mo, Double S, Double S vacancy, Single S vacancy, Mo vacancy
        super().__init__()

        # Encoder
        self.down1 = DoubleConv2d(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv2d(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv2d(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv2d(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = DoubleConv2d(512, 1024)

        # Decode
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = DoubleConv2d(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec2 = DoubleConv2d(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv2d(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec4 = DoubleConv2d(128, 64)

        self.final = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        # Bridge
        b = self.bridge(p4)

        # Decoder
        u1 = self.up1(b)
        cat1 = torch.cat([u1, d4], dim=1)
        dc1 = self.dec1(cat1)

        u2 = self.up2(dc1)
        cat2 = torch.cat([u2, d3], dim=1)
        dc2 = self.dec2(cat2)

        u3 = self.up3(dc2)
        cat3 = torch.cat([u3, d2], dim=1)
        dc3 = self.dec3(cat3)

        u4 = self.up4(dc3)
        cat4 = torch.cat([u4, d1], dim=1)
        dc4 = self.dec4(cat4)

        # Output
        out = self.final(dc4)

        return out

In [ ]:
N_training = 1000 # number of training images

imgs = np.zeros((N_training, 1, 64, 64), dtype=np.float32)
masks = np.zeros((N_training, 5, 64, 64), dtype=np.float32)

print("Generating a training set...")

for i in range(N_training):
    data_g = datasetGenerator(6)
    data_g.latticeDistortion()

    imgs[i, 0, :, :] = data_g.makeImage()
    masks[i, :, :, :] = data_g.makeMask()

print("Done")

imgs_tensor= torch.from_numpy(imgs).float()
masks_tensor = torch.from_numpy(masks).float()

dataset = ClassificationDataset(imgs_tensor, masks_tensor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
model = UNet().to(device)

epochs = 700
Loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_list = np.zeros(epochs, dtype=float)

print("Training start")

model.train()

for epoch in range(epochs):
    loss_per_running = 0

    for img_batch, mask_batch in loader:
        img_batch = img_batch.to(device)
        mask_batch = mask_batch.to(device)
        
        optimizer.zero_grad()

        pred = model(img_batch)
        loss = Loss(pred, mask_batch)
        loss.backward()
        loss_per_running += loss.item()
        optimizer.step()
        
    loss_list[epoch] = loss_per_running/len(loader)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss_list[epoch]}")

torch.save(model.state_dict(), "UNet_parameters.pt")

plt.plot(loss_list)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig("loss_Unet.png")
plt.clf()

In [ ]:
denoisedImage_np = np.load("data/DenoisedImages.npy")
denoisedImage_tensor = torch.from_numpy(np.transpose(denoisedImage_np, (2,0,1))[:, np.newaxis, :, :]).float().to(device) # np.array to tensor

#model = UNet().to(device)
#model.load_state_dict(torch.load("UNet_parameters.pt", weights_only=True, map_location=torch.device('cpu')))
model.eval()
with torch.no_grad():
    mask_tensor = model(denoisedImage_tensor)

mask_np = mask_tensor.cpu().numpy() # tensor to np.array

for idx in range(10):
    fig = plt.figure(figsize=(12, 8))
    
    plt.subplot(231)
    plt.imshow(denoisedImage_np[:, :, idx])
    plt.title("Input image")
    plt.axis("off")
    
    plt.subplot(232)
    plt.imshow(mask_np[idx][0])
    plt.title("Mo")
    plt.axis("off")
    
    plt.subplot(233)
    plt.imshow(mask_np[idx][1])
    plt.title("Double S")
    plt.axis("off")
    
    plt.subplot(234)
    plt.imshow(mask_np[idx][2])
    plt.title("Double S vacancy")
    plt.axis("off")
    
    plt.subplot(235)
    plt.imshow(mask_np[idx][3])
    plt.title("Single S vacancy")
    plt.axis("off")
    
    plt.subplot(236)
    plt.imshow(mask_np[idx][4])
    plt.title("Mo vacancy")
    plt.axis("off")
    
    plt.tight_layout()
    plt.savefig(f"classification_{idx}.png", dpi=300)
